# Week 3 — Meeting minutes on Google Colab

The course runs open models on a Colab GPU. This notebook does the same with this repo's package:

1. Transcribe audio with an open Whisper model on the GPU (Runtime → Change runtime type → T4).
2. Turn the transcript into checked minutes with an open model through the Hugging Face router.

Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left bar). Llama models are gated: accept the licence on the model's Hugging Face page first, or set another model below.

*This notebook has not been run end to end by the author yet; the package itself is covered by offline tests.*

In [ ]:
%pip install -q "git+https://github.com/MOHAMEDMEDOHEMEDA/llm_labs.git" "transformers[torch]"

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
from week3_minutes.transcribe import transcribe

# Upload an audio file with the folder icon on the left, then set its name here.
AUDIO = "meeting.mp3"

transcript = transcribe(AUDIO)  # openai/whisper-small; pass model_id= for a larger one
print(transcript[:1000])

In [ ]:
from IPython.display import Markdown, display

from labkit.providers import PROVIDERS, client_for
from week3_minutes.minutes import OpenAICompatibleLLM, make_minutes, render_markdown, usage_summary

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
provider = PROVIDERS["huggingface"]
llm = OpenAICompatibleLLM(client_for(provider, os.environ["HF_TOKEN"]), MODEL)

minutes = await make_minutes(llm, transcript, title="Meeting minutes")
display(Markdown(render_markdown(minutes)))
print(usage_summary(llm))

Whisper gives text without speaker names, so owners are often missing; items the model can't tie to a named person in the transcript appear under **Needs checking**.